# Deploy de modelos

Nesta aula, vamos aprender como fazer o deploy nosso modelo de recomendação.

**Governança via Unity Catalog:** Gerenciamos versões e permissões no catálogo centralizado.
**Aliases (Champion/Challenger):** Em vez de apontar para a "Versão X", o aplicativo aponta para o "Champion". Isso permite trocar o modelo por trás da API sem alterar o código do aplicativo.

In [0]:
%%capture
## instalando os pacotes
%pip install databricks lightgbm
%pip install databricks-feature-engineering
dbutils.library.restartPython()

#Governança com MlflowClient e Deploy Batch

## O que é o `MlflowClient`?

O `MlflowClient` é a API de mais baixo nível do MLflow. Enquanto funções como `mlflow.start_run()` ou `mlflow.log_model()` lidam com o rastreamento automático da célula atual, o `MlflowClient` interage diretamente com o servidor central de metadados do **Unity Catalog**.

Ele é utilizado para ações administrativas e de governança, tais como:

* Consultar versões existentes de um modelo.
* Criar, deletar ou modificar tags e metadados.
* Mapear e gerenciar **Aliases** (como `@Champion` e `@Challenger`) para desacoplar a engenharia de software da engenharia de machine learning.

### Funções Utilizadas

* **`client.get_registered_model(model_name)`**: Acessa o catálogo e traz um objeto contendo os metadados do modelo de três níveis (`catalogo.esquema.modelo`), incluindo quem o criou, quando e a lista de todas as versões salvas.
* **`client.set_registered_model_alias(model_name, alias, version)`**: Vincula uma tag de produção (*Alias*) à versão. Na arquitetura moderna do Unity Catalog, os estágios antigos (`Staging`/`Production`) foram substituídos por Aliases flexíveis.

**Por que usamos Aliases?** O seu pipeline de escoragem em lote ou a sua API REST não vão apontar para a `Versão 4` fixa. Eles vão apontar para `model_name@Champion`. Quando um modelo novo for treinado e validado, basta mover a tag `@Champion` para a nova versão via Client. Todo o seu ecossistema produtivo se atualiza automaticamente sem que uma única linha de código de engenharia precise ser alterada.

### Como o Spark processa isso por baixo do pano?

1. **`mlflow.pyfunc.spark_udf`**: Transforma o artefato do LightGBM/XGBoost em uma função nativa do Spark SQL. Ele lê a **Signature (Assinatura)** do modelo para mapear quais colunas o Spark deve injetar.
2. **Distribuição nos Nós Workers**: O Spark divide a sua tabela Delta de entrada em múltiplas partições. Cada nó *Worker* do cluster recebe um pedaço dos dados e uma cópia isolada do modelo carregado pela UDF.
3. **Escoragem em Paralelo**: Os nós processam o método `.predict_proba()` simultaneamente em memória de forma massiva, gerando a nova coluna de score sem gargalos no nó *Driver*.
4. **Gravação Delta**: O resultado final pode ser salvo de volta no Unity Catalog usando `.write.mode("overwrite").saveAsTable()`.

In [0]:
from mlflow.tracking import MlflowClient
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql.functions import col

catalogo = "workspace"
esquema = "gold"
model_name = f"{catalogo}.{esquema}.propensao_compra_modelo"

# Criando o cliente do MLflow
client = MlflowClient()

print(f"Buscando as versões do modelo {model_name} no Unity Catalog...")

# Em vez de get_registered_model, buscamos todas as versões associadas a esse nome
versions = client.search_model_versions(f"name='{model_name}'")

# Ordenamos as versões de forma decrescente para garantir que a mais recente fique no topo
latest_version = sorted([int(v.version) for v in versions], reverse=True)[0]

# Agora definimos o Alias usando a versão dinâmica recuperada
client.set_registered_model_alias(model_name, "Champion", str(latest_version))

print(f" Versão {latest_version} definida como @Champion com sucesso!")

In [0]:
import pandas as pd
from pyspark.sql.functions import col, to_date

tabela_novos_dados = f"{catalogo}.{esquema}.base_features_modelo_novos_clientes"

# Simulacao com dados ficticios
dados_validacao_estritos = {
    "id_usuario": [f"usr_val_{i}" for i in range(1, 14)],
    "dia_prtc": ["2026-07-05"] * 13,
    "idade_cliente":               [37, 68, 43, 44, 56, 91, 48, 34, 43, 33, 67, 55, 54],
    "renda_mensal_k":              [1102.2, 1619.8, 2134.9, 6134.9, 1702.9, 1619.8, 1449.1, 1449.1, 1536.8, 728.1, 1244.9, 1200.4, 1489.1],
    "tempo_medio_clique_segundos": [118.0, 119.0, 121.5, 121.5, 146.0, 92.0, 133.0, 123.0, 112.0, 90.0, 130.0, 127.0, 106.0],
    "media_interacoes_suporte":    [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    "media_cupons_ativos":         [1.0, 1.0, 2.0, 2.0, 1.0, 1.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0, 3.0],
    "media_score_nps_cliente":     [8.0, 8.0, 8.0, 8.0, 8.0, 8.0, 9.0, 9.0, 9.0, 8.0, 10.0, 10.0, 9.0],
    "media_dias_inatividade":      [2.0, 1.0, 2.5, 2.5, 4.0, 4.0, 1.0, 3.0, 3.0, 4.0, 1.0, 3.0, 1.0],
    "total_gasto_acumulado_reais": [5200.0, 700.0, 5000.0, 5000.0, 5200.0, 5200.0, 5200.0, 5200.0, 5200.0, 299.0, 5200.0, 5200.0, 5200.0],
    
    # Variáveis Categóricas OHE
    "genero_cliente_m":            [1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    "regiao_cliente_nordeste":     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "regiao_cliente_norte":        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "regiao_cliente_sudeste":      [0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    "regiao_cliente_sul":          [1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1]
}

df_pandas = pd.DataFrame(dados_validacao_estritos)
df_spark = spark.createDataFrame(df_pandas)

# Tipagem estrita de acordo com o esquema do Unity Catalog
df_spark_final = df_spark \
    .withColumn("dia_prtc", to_date(col("dia_prtc"))) \
    .withColumn("idade_cliente", col("idade_cliente").cast("integer")) \
    .withColumn("renda_mensal_k", col("renda_mensal_k").cast("double")) \
    .withColumn("tempo_medio_clique_segundos", col("tempo_medio_clique_segundos").cast("double")) \
    .withColumn("media_interacoes_suporte", col("media_interacoes_suporte").cast("double")) \
    .withColumn("media_cupons_ativos", col("media_cupons_ativos").cast("double")) \
    .withColumn("media_score_nps_cliente", col("media_score_nps_cliente").cast("double")) \
    .withColumn("media_dias_inatividade", col("media_dias_inatividade").cast("double")) \
    .withColumn("total_gasto_acumulado_reais", col("total_gasto_acumulado_reais").cast("double")) \
    .withColumn("genero_cliente_m", col("genero_cliente_m").cast("integer")) \
    .withColumn("regiao_cliente_nordeste", col("regiao_cliente_nordeste").cast("integer")) \
    .withColumn("regiao_cliente_norte", col("regiao_cliente_norte").cast("integer")) \
    .withColumn("regiao_cliente_sudeste", col("regiao_cliente_sudeste").cast("integer")) \
    .withColumn("regiao_cliente_sul", col("regiao_cliente_sul").cast("integer"))

# Ordenação idêntica ao catálogo
colunas_ordenadas = [
    "id_usuario", "dia_prtc", "idade_cliente", "renda_mensal_k", "tempo_medio_clique_segundos",
    "media_interacoes_suporte", "media_cupons_ativos", "media_score_nps_cliente", "media_dias_inatividade",
    "total_gasto_acumulado_reais", "genero_cliente_m", "regiao_cliente_nordeste", "regiao_cliente_norte",
    "regiao_cliente_sudeste", "regiao_cliente_sul"
]
df_spark_final = df_spark_final.select(*colunas_ordenadas)

print(f"Escrevendo dados validados corretos na tabela: {tabela_novos_dados}")
df_spark_final.write.mode("overwrite").saveAsTable(tabela_novos_dados)

print(" Dataset de validação gravado com os valores exatos!")
display(df_spark_final)

## MLflow Spark UDF (`mlflow.pyfunc.spark_udf`)

O `mlflow.pyfunc.spark_udf` é um utilitário do MLflow projetado para carregar um modelo previamente treinado (registrado no MLflow ou Unity Catalog) e convertê-lo em uma **User-Defined Function (UDF)** do Apache Spark.

Sua principal função é permitir que você aplique modelos de Machine Learning diretamente em DataFrames do PySpark, garantindo que a inferência seja distribuída paralelamente por todos os *workers* do seu cluster.

###Casos de Uso Principais

* **Batch Scoring:** Escoragem de grandes volumes de dados (milhões de linhas) de forma rápida e distribuída.
* **Abstração de Frameworks:** Ele usa o "flavor" genérico `pyfunc`, ou seja, o código do PySpark não precisa saber se o modelo por baixo dos panos é uma árvore do Scikit-Learn ou um Booster do LightGBM.

In [0]:
from pyspark.sql.functions import struct, col
import mlflow

model_name = "workspace.gold.propensao_compra_modelo"
model_uri = f"models:/{model_name}@champion"

# Forçamos o retorno como Inteiro (classe 0 ou 1)
predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri=model_uri, result_type="int")

# Remover 'id_usuario' das features que vão para o modelo
features = [c for c in df_spark_final.columns if c not in ["id_usuario", "dia_prtc"]]

primary_key = "id_usuario"

# preparar o dataset para prever
test_features_df = df_spark_final.select(primary_key, *features)

# realizar a predicao
prediction_df = test_features_df.withColumn(
    "vai_comprar", 
    predict_udf(*test_features_df.drop(primary_key).columns)
).select(primary_key, "vai_comprar")

display(prediction_df)